# 🔬 Prototipação de ML - Clima × Dengue
Este notebook é nosso **Laboratório de Experimentação**. 
O objetivo aqui é carregar os dados completos, isolar um município para treinar um modelo de Machine Learning, visualizar os resultados rapidamente e entender como escalar isso para todos os 1.190 municípios da Região Sul.

## 1. Configuração e Imports
Vamos importar as bibliotecas visuais e adicionar a raiz do projeto no sys.path para podermos usar os módulos que já existem na pasta src/.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Configura o matplotlib para gráficos mais bonitos
plt.style.use('ggplot')

# Adiciona a raiz do projeto para podermos importar a pasta src
BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

## 2. Carregamento da Base Completa (Painel)
No script original, você tem uma função muito boa chamada uild_base_panel. Vamos usá-la para carregar o histórico de **todos** os municípios de uma vez, do jeito que será na produção.

In [3]:
from src.processamento.motor_lote_modelagem_v2_por_latitude import build_base_panel

print("Carregando painel completo de dados...")
# Isso vai ler os CSVs das latitudes e juntar clima, dengue e ENSO
painel, lookup = build_base_panel(BASE_DIR)

print(f"\nTotal de registros carregados: {len(painel):,}")
print(f"Total de municípios únicos: {painel['code'].nunique():,}")

# Mostra as primeiras 5 linhas aleatórias
painel.sample(5)

18:49:32 [INFO] build_base_panel → fonte canônica: output/csvs_latitude/ (1191 municípios esperados)


Carregando painel completo de dados...

Total de registros carregados: 142,800
Total de municípios únicos: 1,190


,code,year,month,dengue_cases,latitude,city_name,lat_faixa_id,temp_media,precipitacao,enso
73356,421430,2018,1,0.0,-27.673971,Rancho Queimado,5,21.6,162.2,-0.90
90296,430280,2019,9,0.0,-30.593435,Caçapava Do Sul,7,22.3,97.4,0.20
5008,410320,2022,5,4.0,-23.713029,Bom Sucesso,2,14.9,236.8,-0.99
23832,411430,2021,1,0.0,-25.842355,Mandirituba,4,23.2,222.0,-1.00
61661,420757,2023,6,0.0,-26.987792,Iomerê,4,17.6,178.2,0.77


## 3. Isolando um Município para Estudo
Treinar 1.190 modelos aqui demoraria muito. A regra de ouro do Notebook é: **isole, entenda a matemática num escopo pequeno, plote, e se der certo, escale.** Vamos pegar *Joinville* como exemplo.

In [4]:
# Buscando Joinville no dataset
cidade_alvo = 'joinville'
df_cidade = painel[painel['city_name'].str.lower().str.contains(cidade_alvo, na=False)].copy()

if df_cidade.empty:
    print("Cidade não encontrada. Pegando a cidade com mais casos (pico)...")
    codigo_maior_pico = painel.groupby('code')['dengue_cases'].sum().idxmax()
    df_cidade = painel[painel['code'] == codigo_maior_pico].copy()

# Vamos criar uma coluna de Data real (YYYY-MM-DD) para facilitar o gráfico
df_cidade['data'] = pd.to_datetime(
    df_cidade['year'].astype(str) + '-' + df_cidade['month'].astype(str).str.zfill(2)
)
df_cidade = df_cidade.sort_values('data').reset_index(drop=True)

nome_cidade = df_cidade['city_name'].iloc[0].upper()
print(f"Dados isolados para: {nome_cidade} ({len(df_cidade)} meses de histórico)")

Dados isolados para: JOINVILLE (120 meses de histórico)


### 3.1. Visão Temporal (EDA - Exploratory Data Analysis)
Vamos ver o gráfico de Casos de Dengue x Temperatura ao longo do tempo.

In [5]:
fig, ax1 = plt.subplots(figsize=(14, 6))

# Eixo Y principal: Dengue (Barras vermelhas)
ax1.bar(df_cidade['data'], df_cidade['dengue_cases'], color='indianred', alpha=0.7, width=20, label='Casos Dengue')
ax1.set_ylabel('Casos de Dengue', color='indianred', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='indianred')

# Eixo Y secundário: Temperatura (Linha azul)
ax2 = ax1.twinx()
ax2.plot(df_cidade['data'], df_cidade['temp_media'], color='steelblue', linewidth=2, label='Temperatura Média')
ax2.set_ylabel('Temperatura (°C)', color='steelblue', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='steelblue')

plt.title(f"Série Temporal Climático-Epidemiológica: {nome_cidade}", fontsize=16)
fig.tight_layout()
plt.show()

C:\Users\ruiga\AppData\Local\Temp\ipykernel_4396\344624925.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Feature Engineering: Criando a Memória do Mosquito (Lags)
Como você bem sabe, a dengue reage à chuva/temperatura de meses atrás. Vamos recriar o lag temporal aqui.

In [6]:
# Shift de 3 meses para trás
df_cidade['temp_lag3'] = df_cidade['temp_media'].shift(3)
df_cidade['prec_lag3'] = df_cidade['precipitacao'].shift(3)
df_cidade['enso_lag3'] = df_cidade['enso'].shift(3)

# Removendo as linhas iniciais que ficaram com valor Nulo por causa do shift
df_ml = df_cidade.dropna(subset=['temp_lag3', 'prec_lag3', 'enso_lag3']).copy()

df_ml[['data', 'temp_media', 'temp_lag3', 'dengue_cases']].head(6)

,data,temp_media,temp_lag3,dengue_cases
3,2015-04-01,14.9,21.1,0.0
4,2015-05-01,14.9,19.1,0.0
5,2015-06-01,18.4,17.9,0.0
6,2015-07-01,15.5,14.9,0.0
7,2015-08-01,19.0,14.9,0.0
8,2015-09-01,17.0,18.4,0.0


## 5. Treinando o Modelo (Machine Learning)
Vamos usar o modelo que você já importou (HistGradientBoostingRegressor). 
Separamos o histórico em duas partes: **Treino** (até 2022) e **Teste** (2023-2024).

In [7]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Definir o X (Features/Causas) e o y (Target/Efeito)
features = ['temp_lag3', 'prec_lag3', 'enso_lag3', 'month']
X = df_ml[features]
y = df_ml['dengue_cases']

# 2. Split Temporal
mask_treino = df_ml['year'] <= 2022
mask_teste = df_ml['year'] > 2022

X_treino, y_treino = X[mask_treino], y[mask_treino]
X_teste, y_teste = X[mask_teste], y[mask_teste]

# 3. Instanciar e Treinar o Modelo
modelo = HistGradientBoostingRegressor(random_state=42)
modelo.fit(X_treino, y_treino)

# 4. Fazer Predições no período de Teste
predicoes = modelo.predict(X_teste)

# Evita previsões de casos negativos
predicoes = np.clip(predicoes, 0, None)

df_ml.loc[mask_teste, 'predicao'] = predicoes

print("Treinamento concluído!")
print(f"MAE (Erro Médio Absoluto): {mean_absolute_error(y_teste, predicoes):.1f} casos/mês")
print(f"R² Score: {r2_score(y_teste, predicoes):.2f}")

Treinamento concluído!
MAE (Erro Médio Absoluto): 0.3 casos/mês
R² Score: -0.13


## 6. Verificando a Realidade x Predição
No script .py, você só via os PDFs no final, e às vezes nem via se desse erro no meio. Aqui, você enxerga a avaliação do modelo no mesmo segundo.

In [8]:
df_teste = df_ml[mask_teste]

plt.figure(figsize=(12, 5))
plt.plot(df_teste['data'], df_teste['dengue_cases'], label='Casos REAIS', color='black', linewidth=2, marker='o')
plt.plot(df_teste['data'], df_teste['predicao'], label='Previsão do ML', color='orange', linewidth=2, linestyle='dashed', marker='x')

plt.title(f"Teste do Modelo em Dados Nunca Vistos (2023-2024): {nome_cidade}")
plt.ylabel("Casos de Dengue")
plt.legend()
plt.show()

C:\Users\ruiga\AppData\Local\Temp\ipykernel_4396\2902760788.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Como isso vai para Produção (Escalonamento)?
Veja como é simples: nós pegamos o código que funcionou na cidade alvo (células 5 e 6 acima), colocamos dentro de uma função def treinar_municipio(df_cidade): e mandamos rodar um or cidade in painel['code'].unique():.

Isso é exatamente o que o seu script motor_lote_modelagem_v2_por_latitude.py faz, mas agora você tem total liberdade para explorar, alterar o modelo para Prophet, mudar o LAG para 2 meses e ver os gráficos interativos antes de colocar o robô pra processar 1.190 cidades às cegas!